## Extension Analysis Work (on-going)

In [41]:
from tabulate import tabulate
import pandas as pd

pd.set_option('display.max_rows', None)  
pd.set_option('display.max_columns', None)  
pd.set_option('display.expand_frame_repr', False)

In [42]:
def joules_to_kwh(joules):
    return float(joules) / 3600000

def load_philipp_data(cluster, workflow, runs):
    processed = {}

    for run in range(1, runs+1):
        with open(f'nxf-experiments/{cluster}-cluster/{workflow}/{run}/wf_data.csv') as file:
            stripped_lines = [line.rstrip().split(',') for line in file.readlines()]
            data = stripped_lines[1:]

        pkg = data[0][11]
        dram = data[0][12]
        total = data[0][13]

        processed[run] = {}
        processed[run]['pkg'] = joules_to_kwh(pkg)
        processed[run]['dram'] = joules_to_kwh(dram)
        processed[run]['total'] = joules_to_kwh(total)
    
    return processed


def load_data_totals(cluster, workflow, runs):
    processed = {}

    for run in range(1, runs+1):
        proc_data = pd.read_csv(f'nxf-experiments/{cluster}-cluster/{workflow}/{run}/trace.csv', header=0)
        processed[run] = {}
        processed[run]['read'] = proc_data['read_bytes'].sum() / 1073741824  # convert from B to GiB
        processed[run]['write'] = proc_data['write_bytes'].sum() / 1073741824  # convert from B to GiB

    return processed


def load_philipp_total_energy(cluster, workflow, runs):
    processed = {}
        
    for run in range(1, runs+1):
        with open(f'nxf-experiments/{cluster}-cluster/{workflow}/{run}/task_data.md') as file:
            lines = [line.strip() for line in file.readlines()]

        final = lines[-1].split(': ')[1][:-6]

        processed[run] = {}
        processed[run]['pkg'] = None
        processed[run]['dram'] = None
        processed[run]['total'] = joules_to_kwh(final)
    
    return processed

In [43]:
# Functions to parse rapl data
def load_workflow_rapl_readings(cluster, workflow):
    with open(f'nxf-experiments/{cluster}-cluster/{workflow}-runs.csv', 'r') as file:
        data = [line.strip().split(',') for line in file.readlines()]
    
    return data[1:]

gu_socket = {'rnaseq': '_2', 'chipseq': '_2', 'nanoseq': 'both', 'atacseq': 'both'}

def process_rapl_data(data, gu=False, wf=''):
    processed = {}
    run = 1

    for row in data:
        if gu:
            if gu_socket[wf] == 'both':
                idle_dram_2 = float(row[14])
                total_dram_2 = float(row[11]) 
                active_dram_2 = float(row[17])
                idle_dram_1 = float(row[5])
                total_dram_1 = float(row[2])
                active_dram_1 = float(row[8])
                rapl_pkg = float(row[7]) + float(row[16])

                if idle_dram_1 + idle_dram_2 > total_dram_1 + total_dram_2:
                    rapl_mem = total_dram_1 + total_dram_2
                else:
                    rapl_mem = active_dram_1 + active_dram_2
                rapl_total = rapl_pkg + rapl_mem
            else:
                rapl_pkg = float(row[16])
                idle_dram_2 = float(row[14])
                total_dram_2 = float(row[11]) 
                active_dram_2 = float(row[17])
                if idle_dram_2 > total_dram_2:
                    rapl_mem = total_dram_2
                else:
                    rapl_mem = active_dram_2
                rapl_total = rapl_pkg + rapl_mem
        else:
            rapl_pkg = float(row[7])
            active_dram = float(row[8])
            idle_dram = float(row[5])
            total_dram = float(row[2])

            if idle_dram > total_dram:
                rapl_mem = total_dram
            else:
                rapl_mem = active_dram

            rapl_total = rapl_pkg + rapl_mem

        processed[run] = {}
        processed[run]['pkg'] = rapl_pkg
        processed[run]['dram'] = rapl_mem
        processed[run]['total'] = rapl_total
        run += 1

    return processed

In [44]:
# Functions to parse ichnos data
def load_ichnos_summary_file(summary_file):
    with open(summary_file, 'r') as file:
        raw = [line.strip() for line in file.readlines()]

    cpu = float(raw[7].split(':')[1].strip()[:-3])
    task_mem = float(raw[9].split(':')[1].strip()[:-3])
    node_mem = float(raw[15].split(':')[1].strip()[:-3])

    return (cpu, task_mem, node_mem)

def load_ichnos_data(cluster, workflow, runs, strategy):
    workflow_path = f'nxf-experiments/{cluster}-cluster/{workflow}/ichnos'
    data = {}

    for run in range(1, runs + 1):
        data[run] = {}
        (cpu, task_mem, node_mem) = load_ichnos_summary_file(f'{workflow_path}/{cluster}-{workflow}-{run}-1-{strategy}-summary.txt')
        data[run]['pkg'] = cpu
        data[run]['dram'] = task_mem + node_mem 
        data[run]['total'] = cpu + task_mem + node_mem

    return data

In [45]:
# Generic functions
def get_error(experimental, actual):
    return round((abs(experimental - actual) / actual) * 100, 2)

In [61]:
rapl_data = {'hu': {}, 'gu': {}}
rapl_data['hu']['rnaseq'] = process_rapl_data(load_workflow_rapl_readings('hu', 'rnaseq'))
rapl_data['hu']['chipseq'] = process_rapl_data(load_workflow_rapl_readings('hu', 'chipseq'))
rapl_data['hu']['atacseq'] = process_rapl_data(load_workflow_rapl_readings('hu', 'atacseq'))
rapl_data['hu']['rangeland'] = process_rapl_data(load_workflow_rapl_readings('hu', 'rangeland'))
rapl_data['hu']['nanoseq'] = process_rapl_data(load_workflow_rapl_readings('hu', 'nanoseq'))
rapl_data['hu']['sarek'] = process_rapl_data(load_workflow_rapl_readings('hu', 'sarek'))

rapl_data['gu']['atacseq'] = process_rapl_data(load_workflow_rapl_readings('gu', 'atacseq'), True, 'atacseq')
rapl_data['gu']['chipseq'] = process_rapl_data(load_workflow_rapl_readings('gu', 'chipseq'), True, 'chipseq')
rapl_data['gu']['nanoseq'] = process_rapl_data(load_workflow_rapl_readings('gu', 'nanoseq'), True, 'nanoseq')
rapl_data['gu']['rnaseq'] = process_rapl_data(load_workflow_rapl_readings('gu', 'rnaseq'), True, 'rnaseq')

ichnos_data = {'hu': {}, 'gu': {}}
ichnos_data['hu']['rnaseq'] = load_ichnos_data('hu', 'rnaseq', len(rapl_data['hu']['rnaseq']), 'schedutil_linear')
ichnos_data['hu']['chipseq'] = load_ichnos_data('hu', 'chipseq', len(rapl_data['hu']['chipseq']), 'schedutil_linear')
ichnos_data['hu']['atacseq'] = load_ichnos_data('hu', 'atacseq', len(rapl_data['hu']['atacseq']), 'performance_linear')
ichnos_data['hu']['rangeland'] = load_ichnos_data('hu', 'rangeland', len(rapl_data['hu']['rangeland']), 'schedutil_linear')
ichnos_data['hu']['nanoseq'] = load_ichnos_data('hu', 'nanoseq', len(rapl_data['hu']['nanoseq']), 'performance_linear')
ichnos_data['hu']['sarek'] = load_ichnos_data('hu', 'sarek', len(rapl_data['hu']['sarek']), 'schedutil_linear')

ichnos_data['gu']['rnaseq'] = load_ichnos_data('gu', 'rnaseq', len(rapl_data['gu']['rnaseq']), 'ondemand_linear')
ichnos_data['gu']['atacseq'] = load_ichnos_data('gu', 'atacseq', len(rapl_data['gu']['atacseq']), 'ondemand_linear')
ichnos_data['gu']['chipseq'] = load_ichnos_data('gu', 'chipseq', len(rapl_data['gu']['chipseq']), 'ondemand_linear')
ichnos_data['gu']['nanoseq'] = load_ichnos_data('gu', 'nanoseq', len(rapl_data['gu']['nanoseq']), 'ondemand_linear')

io_data = {'hu': {}, 'gu': {}}
io_data['hu']['rnaseq'] = load_data_totals('hu', 'rnaseq', 3)
io_data['hu']['chipseq'] = load_data_totals('hu', 'chipseq', 3)
io_data['hu']['atacseq'] = load_data_totals('hu', 'atacseq', 3)
io_data['hu']['rangeland'] = load_data_totals('hu', 'rangeland', 3)
io_data['hu']['nanoseq'] = load_data_totals('hu', 'nanoseq', 3)
io_data['hu']['sarek'] = load_data_totals('hu', 'sarek', 3)

io_data['gu']['rnaseq'] = load_data_totals('gu', 'rnaseq', 3)
io_data['gu']['chipseq'] = load_data_totals('gu', 'chipseq', 3)
io_data['gu']['atacseq'] = load_data_totals('gu', 'atacseq', 3)
io_data['gu']['nanoseq'] = load_data_totals('gu', 'nanoseq', 3)

table_data = []
table_head = ['cluster', 'workflow', 'run', 'ichnos_pkg', 'rapl_pkg', 'err_pkg', 'ichnos_mem', 'rapl_mem', 'err_mem', 'ichnos_total', 'rapl_total', 'err_total']

for cluster in ['hu', 'gu']:
    for workflow in rapl_data[cluster].keys():
        for run in range(1, len(rapl_data[cluster][workflow].keys()) + 1):
            rapl_entry = rapl_data[cluster][workflow][run]
            rapl_pkg = rapl_entry['pkg']
            rapl_dram = rapl_entry['dram']
            ichnos_entry = ichnos_data[cluster][workflow][run]
            ichnos_pkg = ichnos_entry['pkg']
            ichnos_dram = ichnos_entry['dram']
            io_entry = io_data[cluster][workflow][run]
            table_data.append([cluster, workflow, run, ichnos_pkg, rapl_pkg, get_error(ichnos_pkg, rapl_pkg), ichnos_dram, rapl_dram, get_error(ichnos_dram, rapl_dram), ichnos_entry['total'], rapl_entry['total'], get_error(ichnos_entry['total'], rapl_entry['total'])])

all_data = pd.DataFrame(table_data, columns=table_head)

# report on the workflow runtime
# avg. length of workflow tasks
# make estimation with nf-co2footprint plugin script for comparison

In [64]:
rows_hu = all_data[all_data['cluster'] == 'hu'].reset_index()
median_rows = rows_hu.sort_values(['workflow', 'rapl_total']).groupby('workflow').nth(1)
print(median_rows.sort_values(['rapl_total']))


    index cluster   workflow  run  ichnos_pkg  rapl_pkg  err_pkg  ichnos_mem  rapl_mem  err_mem  ichnos_total  rapl_total  err_total
13     13      hu    nanoseq    2    0.336850  0.361011     6.69    0.095948  0.176021    45.49      0.432798    0.537031      19.41
8       8      hu    atacseq    3    0.476628  0.473352     0.69    0.140942  0.159331    11.54      0.617570    0.632683       2.39
9       9      hu  rangeland    1    1.554003  2.000600    22.32    0.609913  0.479454    27.21      2.163915    2.480054      12.75
1       1      hu     rnaseq    2    1.708393  1.987028    14.02    0.564375  0.534030     5.68      2.272768    2.521058       9.85
4       4      hu    chipseq    2    3.517664  4.111486    14.44    1.161565  1.050971    10.52      4.679229    5.162458       9.36
16     16      hu      sarek    2    3.844159  4.384565    12.33    0.968210  0.931645     3.92      4.812369    5.316210       9.48


In [65]:
rows_gu = all_data[all_data['cluster'] == 'gu'].reset_index()
median_rows = rows_gu.sort_values(['workflow', 'rapl_total']).groupby('workflow').nth(1)
print(median_rows.sort_values(['rapl_total']))

    index cluster workflow  run  ichnos_pkg  rapl_pkg  err_pkg  ichnos_mem  rapl_mem  err_mem  ichnos_total  rapl_total  err_total
6      24      gu  nanoseq    1    0.581757  0.867767    32.96    0.315203  0.124261   153.66      0.896960    0.992028       9.58
11     29      gu   rnaseq    3    0.991637  0.947446     4.66    0.541885  0.474155    14.28      1.533521    1.421601       7.87
0      18      gu  atacseq    1    0.917041  1.353284    32.24    0.511255  0.174389   193.17      1.428296    1.527673       6.51
3      21      gu  chipseq    1    1.296742  1.271727     1.97    0.705438  0.634605    11.16      2.002180    1.906332       5.03
